# CS293 Final: Fine-Tuning Llama 3.2 1B for CCSS Standard Prediction

**Task:** Given a K-12 math problem, predict which Common Core State Standards (CCSS) it addresses.

**Approach:** Fine-tune Llama 3.2 1B using QLoRA on the MathFish dataset from HuggingFace.

**Pipeline:**
1. Load & prepare MathFish data
2. Evaluate base model (pre-fine-tune) on held-out test set
3. Fine-tune with Unsloth + QLoRA
4. Evaluate fine-tuned model on same test set
5. Compare pre vs post accuracy

## 0. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade datasets huggingface_hub

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load MathFish Dataset

In [ ]:
import json
import requests
from collections import Counter

# Download train and test splits directly as JSONL (avoids pyarrow schema issues)
def load_mathfish_jsonl(split):
    url = f"https://huggingface.co/datasets/allenai/mathfish/resolve/main/{split}.jsonl"
    rows = []
    resp = requests.get(url, stream=True)
    for line in resp.iter_lines():
        if line:
            rows.append(json.loads(line))
    return rows

print("Loading train split...")
train_raw = load_mathfish_jsonl("train")
print(f"Train: {len(train_raw)} problems")

print("Loading test split...")
test_raw = load_mathfish_jsonl("test")
print(f"Test: {len(test_raw)} problems")

In [ ]:
# Inspect a sample
sample = train_raw[0]
print("Keys:", list(sample.keys()))
print("\nID:", sample["id"])
print("\nText (first 300 chars):", sample["text"][:300])
print("\nStandards:", sample["standards"])
print("\nSource:", sample.get("source", "N/A"))

## 2. Prepare Data for Instruction Tuning

In [ ]:
# Extract all unique standards from train set for the prompt
all_standards = set()
for row in train_raw:
    for rel_type, code in row["standards"]:
        all_standards.add(code)

all_standards = sorted(all_standards)
print(f"Unique standard codes: {len(all_standards)}")
print(f"Examples: {all_standards[:10]}")

In [ ]:
def format_example(row):
    """Format a MathFish row as an instruction-tuning pair.
    
    Input: problem text
    Output: comma-separated list of CCSS codes the problem ADDRESSES
    (we filter to 'Addressing' relation type only)
    """
    problem_text = row["text"].strip()
    
    # Get only 'Addressing' standards as ground truth
    addressing = sorted(set(
        code for rel_type, code in row["standards"]
        if rel_type == "Addressing"
    ))
    
    if not addressing:
        return None  # Skip problems with no Addressing labels
    
    standards_str = ", ".join(addressing)
    
    return {
        "id": row["id"],
        "instruction": (
            "You are a math education expert. Given a K-12 math problem, predict which "
            "Common Core State Standards (CCSS) the problem directly addresses. "
            "Return ONLY the standard codes as a comma-separated list (e.g., 3.OA.A.1, 3.OA.A.2). "
            "Do not include any explanation."
        ),
        "input": problem_text,
        "output": standards_str,
        "labels": addressing,
    }

# Process train and test
train_data = [x for x in (format_example(r) for r in train_raw) if x is not None]
test_data = [x for x in (format_example(r) for r in test_raw) if x is not None]

print(f"Train examples (with Addressing labels): {len(train_data)}")
print(f"Test examples (with Addressing labels): {len(test_data)}")
print(f"\nSample train output: {train_data[0]['output']}")

In [ ]:
# Distribution of number of standards per problem
label_counts = Counter(len(d["labels"]) for d in train_data)
print("Standards per problem (train):")
for k in sorted(label_counts):
    print(f"  {k} standards: {label_counts[k]} problems ({label_counts[k]/len(train_data)*100:.1f}%)")

In [ ]:
# Format as chat templates for Llama 3.2 Instruct
def to_chat_format(example):
    """Convert to Llama 3.2 chat format for SFTTrainer."""
    return {
        "conversations": [
            {"role": "system", "content": example["instruction"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

train_chat = [to_chat_format(d) for d in train_data]
print(f"Formatted {len(train_chat)} training conversations")
print(f"\nSample conversation:")
for msg in train_chat[0]["conversations"]:
    print(f"  [{msg['role']}]: {msg['content'][:120]}..." if len(msg['content']) > 120 else f"  [{msg['role']}]: {msg['content']}")

## 3. Load Base Model & Evaluate Pre-Fine-Tune

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)
print("Base model loaded.")

In [ ]:
import re
from tqdm import tqdm

def extract_standards_from_text(text):
    """Parse CCSS codes from model output text."""
    # Match patterns like 3.OA.A.1, K.CC.A.1, 8.EE.C.7a, N-RN.A.1, A-SSE.A.1a, etc.
    pattern = r'\b(?:K|[1-8]|N|A|F|G|S|HS)[.-][A-Z]{1,4}(?:[.-][A-Z])?(?:[.-]\d+[a-z]?)\b'
    found = re.findall(pattern, text)
    # Normalize: replace hyphens with dots for consistency, then deduplicate
    normalized = list(dict.fromkeys(found))  # preserve order, deduplicate
    return normalized

def predict_standards(model, tokenizer, problem_text, max_new_tokens=128):
    """Run inference on a single problem and return predicted standard codes."""
    messages = [
        {"role": "system", "content": (
            "You are a math education expert. Given a K-12 math problem, predict which "
            "Common Core State Standards (CCSS) the problem directly addresses. "
            "Return ONLY the standard codes as a comma-separated list (e.g., 3.OA.A.1, 3.OA.A.2). "
            "Do not include any explanation."
        )},
        {"role": "user", "content": problem_text},
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated tokens
    generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    return extract_standards_from_text(generated), generated

# Quick sanity check
FastLanguageModel.for_inference(model)
pred, raw = predict_standards(model, tokenizer, test_data[0]["input"])
print(f"Problem: {test_data[0]['input'][:200]}...")
print(f"Ground truth: {test_data[0]['labels']}")
print(f"Predicted: {pred}")
print(f"Raw output: {raw}")

In [ ]:
def compute_metrics(predictions, ground_truths):
    """Compute precision, recall, F1, exact match at standard, cluster, and domain levels."""
    
    def to_cluster(code):
        """3.OA.A.1 -> 3.OA.A"""
        parts = code.split(".")
        if len(parts) >= 3:
            return ".".join(parts[:3])
        return code
    
    def to_domain(code):
        """3.OA.A.1 -> 3.OA"""
        parts = code.split(".")
        if len(parts) >= 2:
            return ".".join(parts[:2])
        return code
    
    levels = {
        "standard": lambda x: x,
        "cluster": to_cluster,
        "domain": to_domain,
    }
    
    results = {}
    for level_name, transform in levels.items():
        total_precision = 0
        total_recall = 0
        total_f1 = 0
        exact_matches = 0
        n = len(predictions)
        
        for pred, gt in zip(predictions, ground_truths):
            pred_set = set(transform(c) for c in pred)
            gt_set = set(transform(c) for c in gt)
            
            if not pred_set and not gt_set:
                total_precision += 1
                total_recall += 1
                total_f1 += 1
                exact_matches += 1
                continue
            
            if pred_set and gt_set:
                tp = len(pred_set & gt_set)
                p = tp / len(pred_set)
                r = tp / len(gt_set)
                f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
            elif pred_set:
                p, r, f1 = 0, 0, 0
            else:
                p, r, f1 = 0, 0, 0
            
            total_precision += p
            total_recall += r
            total_f1 += f1
            if pred_set == gt_set:
                exact_matches += 1
        
        results[level_name] = {
            "precision": total_precision / n,
            "recall": total_recall / n,
            "f1": total_f1 / n,
            "exact_match": exact_matches / n,
        }
    
    return results

def print_metrics(results, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    print(f"{'Level':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Exact Match':>12}")
    print("-" * 56)
    for level in ["standard", "cluster", "domain"]:
        m = results[level]
        print(f"{level:<12} {m['precision']:>10.3f} {m['recall']:>10.3f} {m['f1']:>10.3f} {m['exact_match']:>12.3f}")

In [ ]:
# Evaluate base model on full test set
print(f"Evaluating base Llama 3.2 1B on {len(test_data)} test problems...")

FastLanguageModel.for_inference(model)
base_predictions = []
base_raw_outputs = []

for ex in tqdm(test_data):
    pred, raw = predict_standards(model, tokenizer, ex["input"])
    base_predictions.append(pred)
    base_raw_outputs.append(raw)

ground_truths = [ex["labels"] for ex in test_data]
base_results = compute_metrics(base_predictions, ground_truths)
print_metrics(base_results, "BASE MODEL (Pre-Fine-Tune)")

In [ ]:
# Show some example predictions from base model
print("\nSample predictions (base model):")
print("=" * 80)
for i in range(min(5, len(test_data))):
    print(f"\nProblem {i+1}: {test_data[i]['input'][:150]}...")
    print(f"  Ground truth: {test_data[i]['labels']}")
    print(f"  Predicted:    {base_predictions[i]}")
    print(f"  Raw output:   {base_raw_outputs[i][:200]}")

## 4. Fine-Tune with QLoRA

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

# Create HF Dataset from our formatted data
train_dataset = Dataset.from_list(train_chat)
train_dataset = standardize_sharegpt(train_dataset)

def apply_template(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in examples["conversations"]]
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True)
print(f"Training dataset: {len(train_dataset)} examples")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        save_strategy="epoch",
        report_to="none",
    ),
)

print("Trainer ready. Starting fine-tuning...")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Evaluate Fine-Tuned Model

In [ ]:
# Evaluate fine-tuned model on test set
print(f"Evaluating fine-tuned Llama 3.2 1B on {len(test_data)} test problems...")

FastLanguageModel.for_inference(model)
ft_predictions = []
ft_raw_outputs = []

for ex in tqdm(test_data):
    pred, raw = predict_standards(model, tokenizer, ex["input"])
    ft_predictions.append(pred)
    ft_raw_outputs.append(raw)

ft_results = compute_metrics(ft_predictions, ground_truths)
print_metrics(ft_results, "FINE-TUNED MODEL (Post-Fine-Tune)")

In [ ]:
# Show some example predictions from fine-tuned model
print("\nSample predictions (fine-tuned model):")
print("=" * 80)
for i in range(min(5, len(test_data))):
    print(f"\nProblem {i+1}: {test_data[i]['input'][:150]}...")
    print(f"  Ground truth: {test_data[i]['labels']}")
    print(f"  Base model:   {base_predictions[i]}")
    print(f"  Fine-tuned:   {ft_predictions[i]}")

## 6. Comparison: Pre vs Post Fine-Tuning

In [ ]:
import pandas as pd

# Build comparison table
rows = []
for level in ["standard", "cluster", "domain"]:
    for model_name, res in [("Llama 3.2 1B (base)", base_results), ("Llama 3.2 1B (fine-tuned)", ft_results)]:
        rows.append({
            "Model": model_name,
            "Level": level,
            "Precision": round(res[level]["precision"], 3),
            "Recall": round(res[level]["recall"], 3),
            "F1": round(res[level]["f1"], 3),
            "Exact Match": round(res[level]["exact_match"], 3),
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("  COMPARISON: Base vs Fine-Tuned Llama 3.2 1B")
print("=" * 80)
print(df.to_string(index=False))

# Compute deltas
print("\n\nF1 Improvement:")
for level in ["standard", "cluster", "domain"]:
    base_f1 = base_results[level]["f1"]
    ft_f1 = ft_results[level]["f1"]
    delta = ft_f1 - base_f1
    print(f"  {level:<12}: {base_f1:.3f} -> {ft_f1:.3f} (Δ = {delta:+.3f})")

## 7. Error Analysis

In [ ]:
# Categorize errors from fine-tuned model
def analyze_errors(predictions, ground_truths, test_data):
    exact_correct = []
    false_positives = []  # predicted but not in ground truth
    false_negatives = []  # in ground truth but not predicted
    partial_matches = []  # some overlap but not exact
    total_misses = []     # zero overlap
    
    for i, (pred, gt) in enumerate(zip(predictions, ground_truths)):
        pred_set = set(pred)
        gt_set = set(gt)
        
        if pred_set == gt_set:
            exact_correct.append(i)
        elif pred_set & gt_set:  # some overlap
            partial_matches.append(i)
            fps = pred_set - gt_set
            fns = gt_set - pred_set
            if fps:
                false_positives.append((i, fps))
            if fns:
                false_negatives.append((i, fns))
        else:  # zero overlap
            total_misses.append(i)
    
    print(f"Exact match: {len(exact_correct)} ({len(exact_correct)/len(predictions)*100:.1f}%)")
    print(f"Partial match: {len(partial_matches)} ({len(partial_matches)/len(predictions)*100:.1f}%)")
    print(f"Total miss: {len(total_misses)} ({len(total_misses)/len(predictions)*100:.1f}%)")
    print(f"\nFalse positive cases: {len(false_positives)}")
    print(f"False negative cases: {len(false_negatives)}")
    
    # Show worst misses
    print(f"\n--- Total Misses (first 5) ---")
    for idx in total_misses[:5]:
        print(f"  Problem: {test_data[idx]['input'][:100]}...")
        print(f"  GT: {ground_truths[idx]}, Pred: {predictions[idx]}")
    
    return exact_correct, partial_matches, total_misses

print("BASE MODEL ERRORS:")
print("-" * 40)
_ = analyze_errors(base_predictions, ground_truths, test_data)

print("\n\nFINE-TUNED MODEL ERRORS:")
print("-" * 40)
_ = analyze_errors(ft_predictions, ground_truths, test_data)

In [ ]:
# Cross-domain confusion analysis for fine-tuned model
def domain_confusion(predictions, ground_truths):
    """Count how often predicted domains differ from ground truth domains."""
    confusion = Counter()
    for pred, gt in zip(predictions, ground_truths):
        pred_domains = set(c.split(".")[1] if len(c.split(".")) >= 2 else c for c in pred)
        gt_domains = set(c.split(".")[1] if len(c.split(".")) >= 2 else c for c in gt)
        
        for pd_ in pred_domains - gt_domains:
            for gd in gt_domains:
                confusion[(gd, pd_)] += 1
    
    print("Cross-domain errors (GT domain -> Predicted domain):")
    for (gt_d, pred_d), count in confusion.most_common(15):
        print(f"  {gt_d} -> {pred_d}: {count}")

print("Fine-tuned model cross-domain confusion:")
domain_confusion(ft_predictions, ground_truths)

## 8. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("llama-1b-ccss-lora")
tokenizer.save_pretrained("llama-1b-ccss-lora")
print("LoRA adapters saved to llama-1b-ccss-lora/")

# Save results for the report
import json

results_out = {
    "base_model": base_results,
    "finetuned_model": ft_results,
    "num_test": len(test_data),
    "num_train": len(train_data),
}

with open("evaluation_results.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Results saved to evaluation_results.json")